# Poker Engine Data Table Analysis

This notebook analyzes the precomputed data tables used by the `submission67` poker agent. The agent relies on several large lookup tables to make fast decisions at runtime. We will explore how they work, visualize their contents, and highlight mathematical flaws or inaccuracies in their construction.

## 1. Environment Setup & Imports
First, we set up the Python path to allow importing from the `submission67` package and load our analysis libraries.

In [ ]:
import os
import sys
import struct
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Set seaborn style for better plots
sns.set_theme(style="whitegrid")

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Map 'submission' to 'submission67' so internal imports work
if 'submission' not in sys.modules:
    import submission67
    sys.modules['submission'] = submission67

from submission67.data_path import get_data_dir
data_dir = get_data_dir()
print(f"Using data directory: {data_dir}")

## 2. Hand Evaluation Tables (`hand_rank_5.npy`, `hand_rank_7.npy`)

### How it works
To avoid evaluating poker hands at runtime, the agent precomputes the rank of every possible hand. The 27-card deck means there are `C(27, 5) = 80,730` unique 5-card hands, and `C(27, 7) = 888,030` unique 7-card combinations.

Cards are encoded as `suit * 9 + rank`. To look up a hand, it is sorted `c0 < c1 < ... < c_n`, and a **combinatorial index** is calculated: `Index = C(c0, 1) + C(c1, 2) + ... + C(cn, n)`. This maps every combination bijectively to a dense array index.

The arrays store integer ranks where **lower is better**. The base values correspond to hand types (e.g., Flushes are in the `200,000`s, Two Pairs are in the `500,000`s).

In [ ]:
# Load the evaluation arrays
rank_5_path = os.path.join(data_dir, "hand_rank_5.npy")
rank_7_path = os.path.join(data_dir, "hand_rank_7.npy")

rank_5 = np.load(rank_5_path)
rank_7 = np.load(rank_7_path)

print(f"5-card table shape: {rank_5.shape}, memory: {rank_5.nbytes / 1e6:.2f} MB")
print(f"7-card table shape: {rank_7.shape}, memory: {rank_7.nbytes / 1e6:.2f} MB")

# Define the base rank brackets (from hand_eval.py)
brackets = {
    'Straight Flush': (0, 100_000),
    'Full House': (100_000, 200_000),
    'Flush': (200_000, 300_000),
    'Straight': (300_000, 400_000),
    'Trips': (400_000, 500_000),
    'Two Pair': (500_000, 600_000),
    'One Pair': (600_000, 700_000),
    'High Card': (700_000, 800_000)
}

def get_category_counts(rank_array):
    counts = {}
    for name, (low, high) in brackets.items():
        mask = (rank_array >= low) & (rank_array < high)
        counts[name] = np.sum(mask)
    return counts

counts_5 = get_category_counts(rank_5)
counts_7 = get_category_counts(rank_7)

df_counts = pd.DataFrame({'5-Card': counts_5, '7-Card': counts_7})
df_counts.plot(kind='bar', figsize=(10, 5), title="Distribution of Hand Types in the 27-Card Deck")
plt.ylabel("Number of Combinations")
plt.yscale('log')
plt.show()
df_counts

Notice that in a 27-card deck (3 suits), a Flush is actually *harder* to get than a Full House in 5-card combinations! This changes standard poker hand rankings.

## 3. Preflop Tables (`preflop_equities.npy`, `preflop_actions.npy`)

### How it works
Similar to the hand evaluation tables, there are 80,730 possible 5-card starting hands. The `preflop_equities.npy` stores the expected win rate (0.0 to 1.0) of each hand against a random opponent hand. `preflop_actions.npy` maps these equities into three buckets: FOLD (1), CALL (2), and RAISE (3).

### Accuracy & Flaws (Monte Carlo Variance)
These tables were generated using 50,000 Monte Carlo rollouts per hand. The standard error for an equity of 0.5 with $n=50,000$ is approximately $\sqrt{0.5 \times 0.5 / 50000} \approx 0.0022$ ($0.22\%$). Because the action buckets use *rigid* thresholds (`0.4727` and `0.5184`), hands whose true equity falls within $3\sigma$ (about $\pm 0.66\%$) of these thresholds have a high chance of being placed in the wrong action bucket due purely to MC noise.

In [ ]:
pf_eq_path = os.path.join(data_dir, "preflop_equities.npy")
pf_act_path = os.path.join(data_dir, "preflop_actions.npy")

pf_eq = np.load(pf_eq_path)
pf_act = np.load(pf_act_path)

THRESH_FOLD_CALL = 0.4727
THRESH_CALL_RAISE = 0.5184
MC_NOISE_3SIGMA = 0.0066  # 3 std devs at 50k rollouts

plt.figure(figsize=(12, 6))
sns.histplot(pf_eq, bins=200, kde=True)
plt.axvline(THRESH_FOLD_CALL, color='red', linestyle='--', label=f'FOLD/CALL ({THRESH_FOLD_CALL})')
plt.axvline(THRESH_CALL_RAISE, color='green', linestyle='--', label=f'CALL/RAISE ({THRESH_CALL_RAISE})')

# Highlight the "danger zones" where MC noise causes mis-bucketing
plt.axvspan(THRESH_FOLD_CALL - MC_NOISE_3SIGMA, THRESH_FOLD_CALL + MC_NOISE_3SIGMA, color='red', alpha=0.2, label='Noise Zone (Mis-bucketed)')
plt.axvspan(THRESH_CALL_RAISE - MC_NOISE_3SIGMA, THRESH_CALL_RAISE + MC_NOISE_3SIGMA, color='green', alpha=0.2)

plt.title("Preflop Equities Distribution & Action Thresholds")
plt.xlabel("Equity")
plt.ylabel("Count")
plt.legend()
plt.show()

misbucketed_estimate = np.sum((np.abs(pf_eq - THRESH_FOLD_CALL) < MC_NOISE_3SIGMA) | (np.abs(pf_eq - THRESH_CALL_RAISE) < MC_NOISE_3SIGMA))
print(f"Total starting hands: {len(pf_eq)}")
print(f"Hands within the 3-sigma MC noise boundary: {misbucketed_estimate} (approx {misbucketed_estimate/len(pf_eq)*100:.1f}%% of all hands are likely assigned the wrong action)")

## 4. BB Discard Table (`bb_discard_table.bin`)

### How it works
This is the largest table (hundreds of MBs). It maps an 8-card combination (5 hole + 3 flop) to the equities of the 10 possible ways to keep 2 cards. It uses **suit isomorphism** (canonicalization) to group identical hands, storing an 8-byte key and 10 `uint8` equity values.

### Accuracy & Flaws
1. **Quantization Loss:** Equities are stored as `uint8` ($0-255$), meaning the actual float equity is `value / 255.0`. This restricts the table to a resolution of $\approx 0.39\%$. If two keep-pairs have true equities of $0.501$ and $0.503$, the table stores them both identically as `128`. The agent might randomly pick the worse keep because it can't tell the difference.
2. **Information Asymmetry Flaw:** The equity was calculated assuming a *random* SB hand. However, at runtime, the SB sees the BB's discards before making their own discard decision. The SB's range is heavily optimized, so the BB table systematically overestimates its own true equity.

In [ ]:
from submission67.player import _load_bb_table, KEEP2

# Note: The file might actually be named flop_table.bin, our fix_data.py script copies it.
bb_table = _load_bb_table()

if bb_table is not None:
    print(f"Loaded BB Table with {len(bb_table._keys):,} entries.")
    
    # Sample the first 10,000 entries to check quantization impacts
    sample_equities = bb_table._equities[:10000]
    
    # How often are the top 2 keep choices tied due to quantization?
    ties = 0
    for row in sample_equities:
        sorted_row = np.sort(row)
        if sorted_row[-1] == sorted_row[-2]:
            ties += 1
            
    print(f"In a sample of 10,000 hands, the top 2 keep-pairs have IDENTICAL quantized equities {ties} times ({ties/10000*100:.1f}%).")
    print("This shows quantization loss directly impacting the agent's ability to choose the strict best pair.")
else:
    print("BB Table not found. Please ensure bb_discard_table.bin exists in the data directory.")

## 5. Equity Percentiles (`equity_percentiles_*.npy`)

### How it works
These arrays define the boundaries for grouping continuous equity values into discrete buckets. This is used by the Deep CFR (DCFR) abstraction system. By looking at the size of these arrays, we can see how many buckets the abstraction uses for each street.

In [ ]:
streets = ['preflop', 'flop', 'turn', 'river']
percentile_lengths = {}

for s in streets:
    path = os.path.join(data_dir, f"equity_percentiles_{s}.npy")
    if os.path.exists(path):
        arr = np.load(path)
        percentile_lengths[s] = len(arr)
        
plt.figure(figsize=(8, 4))
plt.bar(percentile_lengths.keys(), percentile_lengths.values())
plt.title("DCFR Abstraction Granularity (Buckets per Street)")
plt.ylabel("Number of Buckets")
plt.show()

print("Number of buckets by street:", percentile_lengths)
print("Notice how the abstraction becomes coarser (fewer buckets) on the later streets. Usually, CFR abstractions are finer on later streets, but this agent might use a real-time solver on the river instead, relying less on the blueprint abstraction.")